In [1]:
import pandas as pd
from pyprojroot import here

In [2]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [3]:
crime = pd.read_csv(RAW_DATA_DIR/"crime/Municipal-Delitos-2015-2025_abr2026.csv", encoding='latin-1')
population = pd.read_csv(INTERIM_DATA_DIR/"population_2015_2025.csv")

In [4]:
crime['CVEGEO'] = crime['Cve. Municipio'].astype(str).str.zfill(5)
population['CVEGEO'] = population['CVEGEO'].astype(str).str.zfill(5)

In [5]:
crime = crime.rename(columns={'Año': 'year'})
crime['total'] = crime.iloc[:, 9:21].sum(axis=1)

In [6]:
A_crime = [ # Criminal activity associated with organized crime by the Federal Law Against Organized Crime
    'Secuestro',
    'Trata de personas',
    'Narcomenudeo',
    'Falsificación',
    'Corrupción de menores'
]

B_crime = [ # Criminal activity associated with organized crime by its general activity
    'Homicidio doloso',
    'Extorsión',
    'Evasión de presos'
]

C_crime = [ # Logistics related crime
    'Robo a transportista',
    'Robo de vehículo automotor'
]

D_crime = [
    'Delitos cometidos por servidores públicos',
    'Electorales'
]

In [7]:
A_df = crime[crime['Subtipo de delito'].isin(A_crime)].copy()
AB_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime)].copy()
ABC_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime)].copy()
ABCD_df = crime[crime['Subtipo de delito'].isin(A_crime + B_crime + C_crime + D_crime)].copy()

In [8]:
grouped = AB_df.groupby(['year', 'CVEGEO'])['total'].sum().reset_index()
result = grouped.merge(population, on=['year', 'CVEGEO'], how='left')
result.drop(columns='Unnamed: 0', inplace=True)
result['crime_rate'] = result['total'] / result['population'] * 100000

In [9]:
result.to_csv(INTERIM_DATA_DIR/"crime_rates.csv", encoding='latin-1')

print(f" {len(result)} municipalities processed")
print(result.sample(10))

 26153 municipalities processed
       year CVEGEO  total  population  crime_rate
18890  2023  07110      0     15536.3    0.000000
20507  2023  22005     51    139687.8   36.509989
2331   2016  12034     11     73867.4   14.891549
22006  2024  16029      2     23118.0    8.651267
12659  2020  20385     18    200806.0    8.963876
6395   2018  05036      8     41139.6   19.445984
9298   2019  13047      1      9568.8   10.450631
10041  2019  20238      0      1952.4    0.000000
22868  2024  21100      0     18312.6    0.000000
17366  2022  20135      1     16904.2    5.915690
